# Exploration notebook (read-only)

Reads pipeline outputs; never writes back to `data/` or `analysis/`. The reproducible pipeline stays in `scripts/`.

- **A** Step 2 QA: check segmentation
- **B** Step 5b: walk through human-LLM disagreements (only AFTER blind coding is finished)
- **C** Step 7: paragraphs around key events for close reading

In [ ]:
from pathlib import Path
import pandas as pd
pd.set_option('display.max_colwidth', 300)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA, ANALYSIS = ROOT / 'data', ROOT / 'analysis'

## A. Segmentation QA (step 2)

In [ ]:
segs = pd.read_csv(DATA / 'segments.csv')
segs.groupby(['company', 'doc_id']).agg(n=('segment_id', 'size'), median_chars=('n_chars', 'median'),
                                        max_chars=('n_chars', 'max'))

In [ ]:
# Read one document top to bottom. Change DOC to each doc_id in turn.
DOC = segs['doc_id'].iloc[0]
segs.loc[segs['doc_id'] == DOC, ['segment_id', 'n_chars', 'text']]

In [ ]:
# Suspects: very long segments (likely merged) and very short ones (likely cut or leftover menus)
pd.concat([segs.nlargest(5, 'n_chars'), segs.nsmallest(5, 'n_chars')])[['segment_id', 'n_chars', 'text']]

## B. Disagreements (step 5b): run only after manual_coding.csv is fully coded

In [ ]:
dis = pd.read_csv(ANALYSIS / 'disagreements.csv')
dis.groupby(['code']).agg(n=('segment_id', 'size'), human_only=('human', 'sum'), llm_only=('llm', 'sum'))

In [ ]:
# One code at a time. For each row decide: LLM wrong / me wrong / definition unclear
CODE = 'AUTH_PROF'
dis.loc[dis['code'] == CODE, ['segment_id', 'human', 'llm', 'llm_rationale', 'text']]

## C. Event windows for close reading (step 7)

In [ ]:
coded = pd.read_csv(DATA / 'coded_llm.csv')
events = pd.read_csv(DATA / 'events.csv', parse_dates=['date'])
df = segs.merge(coded, on='segment_id', how='inner')
df['date'] = pd.to_datetime(df['date'], errors='coerce')
events[['event_id', 'date', 'label']]

In [ ]:
# Paragraphs within +/- DAYS of an event, optionally filtered to one code
EVENT, DAYS, CODE = 'E3', 30, None   # e.g. CODE = 'REBUTTAL'
d = events.set_index('event_id').loc[EVENT, 'date']
win = df[(df['date'] - d).abs() <= pd.Timedelta(days=DAYS)]
if CODE:
    win = win[win[CODE] == 1]
win[['segment_id', 'company', 'date', 'text']].sort_values('date')